# Category 9 - Capture Coach Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares frame-quality gating strategies for the expensive vision path in Ally Vision v2. The current project uses a lightweight heuristic gate because bad frames waste money, add latency, and produce confusing assistance for blind users.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Capture coach code": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py",
    "framing_judge code": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py",
    "OpenCV Laplacian tutorial": "https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html",
    "AWS Rekognition image features": "https://aws.amazon.com/rekognition/image-features/",
    "Camera capture hook": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/frontend/hooks/useCameraCapture.ts"
  },
  "comparison_rows": [
    {
      "Metric": "Added API cost",
      "Capture Coach (Ally)": "No [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "No [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "Yes, requires model call [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "No [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "Yes [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "No [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "No [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "No if local, yes if cloud [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "No if local, yes if cloud [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "No [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "No [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Works offline",
      "Capture Coach (Ally)": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "Yes [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "No [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "Yes [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "No [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "Yes [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "Yes [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "Yes if local [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "Yes if local [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "Yes [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "Yes [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Blur detection method",
      "Capture Coach (Ally)": "Contrast / uniform-frame heuristic [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "Laplacian second derivative [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "N/A (not publicly available) [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "No-reference IQA model [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "Natural-scene-statistics IQA [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "Semantic zero-shot classification [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "Detection confidence / object framing [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "Landmark / heuristic family [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "On-device vision heuristics [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Brightness / exposure check",
      "Capture Coach (Ally)": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "Image properties include brightness / sharpness [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "N/A (not publicly available) [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "N/A (not publicly available) [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "N/A (not publicly available) [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "Possible app-defined [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "Possible app-defined [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Subject centering check",
      "Capture Coach (Ally)": "No explicit centering model [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "N/A (not publicly available) [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "N/A (not publicly available) [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "N/A (not publicly available) [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "Possible with prompt engineering [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "Yes, detection boxes can drive centering [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "N/A (not publicly available) [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "N/A (not publicly available) [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Glare / overexposure detection",
      "Capture Coach (Ally)": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "Possible via image properties family [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "Indirect quality degradation signal [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "Indirect quality degradation signal [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "N/A (not publicly available) [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "N/A (not publicly available) [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "N/A (not publicly available) [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Resolution / size enforcement",
      "Capture Coach (Ally)": "Yes, explicit minimum frame size checks [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "No [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "No explicit byte-size enforcement [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "N/A (not publicly available) [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "N/A (not publicly available) [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "N/A (not publicly available) [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "N/A (not publicly available) [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "N/A (not publicly available) [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "N/A (not publicly available) [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Inference time (qualitative)",
      "Capture Coach (Ally)": "Very low, pure local heuristic [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "Low [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "Network + service latency [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "Low to medium [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "Low to medium [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "Medium to high [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "N/A (not publicly available) [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "N/A (not publicly available) [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Works on mobile browsers",
      "Capture Coach (Ally)": "Yes, backend-side heuristic paired with browser capture [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "N/A (not publicly available) [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "N/A (not publicly available) [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "N/A (not publicly available) [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "N/A (not publicly available) [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "Yes, browser/mobile support family [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "No, Apple platform-specific [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "Feedback to blind user",
      "Capture Coach (Ally)": "Voice guidance already integrated in route [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "No guidance [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "Possible text guidance [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "N/A (not publicly available) [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "N/A (not publicly available) [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "N/A (not publicly available) [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "App-defined guidance [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "N/A (not publicly available) [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "N/A (not publicly available) [source: https://developer.apple.com/documentation/vision]"
    },
    {
      "Metric": "False-positive risk note",
      "Capture Coach (Ally)": "Low-complexity heuristic may reject some acceptable frames [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py]",
      "No quality filter": "No rejection [source: https://developer.mozilla.org/en-US/docs/Web/API/HTMLCanvasElement/toDataURL]",
      "framing_judge.py": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py]",
      "OpenCV Laplacian blur gate": "N/A (not publicly available) [source: https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html]",
      "Cloud vision quality check": "Depends on external model judgement [source: https://aws.amazon.com/rekognition/image-features/]",
      "BRISQUE": "May not align with assistive-task usability [source: https://learnopencv.com/image-quality-assessment-brisque/]",
      "NIQE": "May not align with assistive-task usability [source: https://data-quality-metrics.readthedocs.io/en/latest/image_quality/NIQE.html]",
      "CLIP zero-shot framing check": "N/A (not publicly available) [source: https://github.com/openai/CLIP]",
      "YOLOv8 framing gate": "N/A (not publicly available) [source: https://docs.ultralytics.com/models/yolov8/]",
      "MediaPipe quality estimator": "N/A (not publicly available) [source: https://developers.google.com/mediapipe]",
      "Apple Vision Framework": "N/A (not publicly available) [source: https://developer.apple.com/documentation/vision]"
    }
  ],
  "criteria": [
    "offline",
    "low cost",
    "multi-factor checks",
    "user guidance"
  ],
  "fit_matrix": {
    "Capture Coach (Ally)": [
      1,
      1,
      1,
      1
    ],
    "No quality filter": [
      1,
      1,
      0,
      0
    ],
    "framing_judge.py": [
      0,
      0,
      0.7,
      0.6
    ],
    "OpenCV Laplacian blur gate": [
      1,
      1,
      0.3,
      0
    ],
    "Cloud vision quality check": [
      0,
      0,
      0.8,
      0.3
    ],
    "BRISQUE": [
      1,
      1,
      0.5,
      0
    ],
    "NIQE": [
      1,
      1,
      0.5,
      0
    ],
    "CLIP zero-shot framing check": [
      1,
      0.5,
      0.7,
      0.2
    ],
    "YOLOv8 framing gate": [
      1,
      0.5,
      0.8,
      0.4
    ],
    "MediaPipe quality estimator": [
      1,
      1,
      0.6,
      0.2
    ],
    "Apple Vision Framework": [
      1,
      1,
      0.6,
      0.2
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
labels=list(data['fit_matrix'].keys())
values=[sum(v) for v in data['fit_matrix'].values()]
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 9 - Capture Coach Comparison Overall Fit Score")
ax.bar(labels,values,color=colors); ax.set_ylabel('Derived fit score',color='white'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(charts_dir / 'category9_capture_coach_comparison_chart1.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["Capture Coach (Ally)", "YOLOv8 framing gate", "Cloud vision quality check", "BRISQUE", "NIQE"]; x=np.arange(len(criteria)); width=0.8/len(selected)
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 9 - Capture Coach Comparison Top-5 Criteria View")
for idx,label in enumerate(selected):
 vals=data['fit_matrix'][label]; color=ALLY if label==selected[0] else (DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else COMP); ax.bar(x+(idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white'); plt.tight_layout(); plt.savefig(charts_dir / 'category9_capture_coach_comparison_chart2.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["Capture Coach (Ally)", "YOLOv8 framing gate", "Cloud vision quality check", "BRISQUE", "NIQE"]; mat=np.array([data['fit_matrix'][k] for k in selected])
fig,ax=plt.subplots(figsize=(10,5)); ax.set_facecolor(BG); ax.figure.set_facecolor(BG); im=ax.imshow(mat,cmap='Blues',aspect='auto'); ax.set_title('Ally Vision v2 — Category 9 - Capture Coach Comparison Trade-off Heatmap',color='white'); ax.set_xticks(np.arange(len(criteria))); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.set_yticks(np.arange(len(selected))); ax.set_yticklabels(selected,color='white')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]): ax.text(j,i,f"{mat[i,j]:.0f}",ha='center',va='center',color='white')
plt.colorbar(im); plt.tight_layout(); plt.savefig(charts_dir / 'category9_capture_coach_comparison_chart3.png',dpi=150,bbox_inches='tight'); plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Capture coach code | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/capture_coach.py | brightness, blur, size checks and guidance |
| 2 | framing_judge code | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/vision/framing_judge.py | future model-based framing path |
| 3 | OpenCV Laplacian tutorial | https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html | classic blur/edge gate reference |
| 4 | AWS Rekognition image features | https://aws.amazon.com/rekognition/image-features/ | image brightness, sharpness, and contrast support |
| 5 | Camera capture hook | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/frontend/hooks/useCameraCapture.ts | JPEG compression and size-aware capture path |


## CONCLUSION

Capture Coach wins because it solves the right problem at the lowest operational cost: reject obviously bad frames locally and early, and translate that decision into usable spoken guidance. That is more important for Ally Vision v2 than maximizing theoretical image-quality sophistication.

→ Chosen for Ally Vision v2 ✅
